In [1]:
import gurobipy as gp
import gurobi_utils as gu
import miplib_loader as ml
import example_loader as el
import numpy as np

In [2]:
# A function to create cuts given a target point
def add_balas_ball_cut(relaxed: gp.Model, target, integer_vars, integer_idx, plotter):
    # for each column in the tableau
    # construct a sparse vector for it
    # get the length of that vector via norm1 (plus 1 if we're an int column)
    # add our cut: sum_j(x_j/a_j)
    
    norm = 2
    current = integer_vars.X
    radius = np.linalg.norm(current - target, norm)
    if radius <= relaxed.params.FeasibilityTol:
        return False  # TODO: tolerance should apply to each component individually?
    
    print("   Gap to target:", radius, ":", current[:7], "to", target[:7])
    if plotter is not None:
        plotter.add_ball(current, radius)
    
    basis = gu.read_basis(relaxed)
    tableau, col_to_var, negated_vars = gu.read_tableau(relaxed, basis, extra_rows=1)
    tableau[negated_vars, :] *= -1
    
    # drop the rows of non-integer variables:
    to_drop = [i for i, base in enumerate(basis) if base not in integer_idx]
    tableau = np.delete(tableau, to_drop, axis=0)

    # Conforti has negative vectors with 1 at row=col, with the rest negated.
    # However, empirically, it seems that the opposite is what we really want (gurobi-specific or standardization issue)
    int_cols = [i for i, c in enumerate(col_to_var) if c in integer_idx]
    tableau[-1, int_cols] = -1  # use our extra row to store the col==row -> 1
  
    lengths = np.linalg.norm(tableau, norm, axis=0)
    lengths /= radius
        
    variables = relaxed.getVars()  # TODO: pass this is as it's expensive
    constraints = relaxed.getConstrs()
    def find_variable(index):
        if index < len(variables):
            return variables[index]
        # if only gurobi gave us access to their slack variables...
        # instead, we have to solve for it:
        cons_idx = index - len(variables)
        constraint = constraints[cons_idx]
        if constraint.Sense != '>':
            assert constraint.Sense == '='
            return 0.0  # ignore equality constraints with slacks (as gurobi generates a slack for every constraint)
        lhs, rhs = relaxed.getRow(constraint), constraint.RHS
        # if cons_idx in negated_vars:
        #     return rhs - lhs
        return lhs - rhs
    
    summed_terms = gp.quicksum(lengths[i] * find_variable(j) for i, j in enumerate(col_to_var))
    if plotter is not None:
        plotter.add_constraint(summed_terms >= 1)
    relaxed.addConstr(summed_terms >= 1)
    return True

# a function to run cuts against the nearest integer:
def run_cuts_to_nearest_int(instances, cut_function, loops=8):
    for instance in instances:
        model: gp.Model = instance.as_gurobi_model()
        print("Running model:", model.ModelName)
        model.params.LogToConsole = 0
        model.params.Method = 1
        gu.standardize_lt_to_gt(model)
        int_vars, int_idx = gu.relax_int_or_bin_to_continuous(model)
        plotter = pu.create()
        for i in range(loops):
            model.optimize()
            if model.Status != gp.GRB.OPTIMAL:
                print("   FAILED! Status:", model.Status)
                break
            nearest = gu.nearest_integer(int_vars)
            if not cut_function(model, nearest, int_vars, int_idx, plotter):
                print("   Final score of relaxed:", model.getObjective().getValue())
                break
        plotter.render()

# test the cuts on simple examples:
run_cuts_to_nearest_int(el.get_instances().values(), add_balas_ball_cut)

Set parameter Username
Academic license - for non-commercial use only - expires 2024-10-05
Set parameter Presolve to value 0
Set parameter Heuristics to value 0
Set parameter Cuts to value 0
Set parameter Presolve to value 0
Set parameter Heuristics to value 0
Set parameter Cuts to value 0
Set parameter Presolve to value 0
Set parameter Heuristics to value 0
Set parameter Cuts to value 0
Set parameter Presolve to value 0
Set parameter Heuristics to value 0
Set parameter Cuts to value 0
Running model: 2D from bottom
   Negated 2 constraints on 2D from bottom
   Relaxed 2 variables on 2D from bottom
   Gap to target: 0.5555555555555558 : [1.22222222 2.33333333] to [1. 2.]
   Gap to target: 0.11111111111111105 : [0.94444444 2.05555556] to [1. 2.]
   Gap to target: 0.0888888888888888 : [1.04444444 2.04444444] to [1. 2.]
   Gap to target: 0.16000000000000147 : [1.12444444 2.03555556] to [1. 2.]
   Gap to target: 0.04938271604939248 : [0.97530864 2.02469136] to [1. 2.]
   Gap to target: 0.04

In [4]:
# a function to find the hyperplane closest to a point
def compute_hyperplane_via_lp(x0, b0, tableau, basis, col_to_var, int_idx):
    m = gp.Model()
    b = m.addVar(lb=-gp.GRB.INFINITY)
    w = m.addMVar(x0.shape, lb=-gp.GRB.INFINITY)
    wn = m.addVar()
    m.addConstr(wn == gp.norm(w, 1))
    m.addConstr(wn == 1)  # optional
    z = m.addVar()
    m.addConstr(z >= x0 @ w - b - b0)
    m.addConstr(z >= -x0 @ w + b + b0)
    m.setObjective(z)
    # our w represents all integer variables,
    # so not all columns in the tableau have a corresponding integer var.
    for i, vec in enumerate(tableau.T):
        cv = col_to_var[i]
        w_idx = int_idx.get(cv, -1)
        if w_idx >= 0:
            m.addConstr(vec[:-1] @ w[basis] + vec[-1] * w[w_idx] <= b)
        else:
            assert vec[-1] == 0.0
            m.addConstr(vec[:-1] @ w[basis] <= b)
    m.params.LogToConsole = 0
    m.optimize()
    assert m.Status == gp.GRB.OPTIMAL
    return w.X, b.X

# create a function to do cuts via LP:
def add_lp_ball_cut(relaxed: gp.Model, target, integer_vars, integer_idx):
    
    norm = 1
    current = integer_vars.X
    radius = np.linalg.norm(current - target, norm)
    if radius <= relaxed.params.FeasibilityTol:
        return False  # TODO: tolerance should apply to each component individually?
    
    print("   Gap to target:", radius, ":", current[:7], "to", target[:7])
    
    basis = gu.read_basis(relaxed)
    tableau, col_to_var, negated_vars = gu.read_tableau(relaxed, basis, extra_rows=1)
    
    # drop the rows of non-integer variables:
    to_drop = [i for i, base in enumerate(basis) if base not in integer_idx]
    tableau = np.delete(tableau, to_drop, axis=0)
    basis = np.delete(basis, to_drop)
    
    # integer_idx goes from var num to index in integer_vars
    int_cols = [i for i, c in enumerate(col_to_var) if c in integer_idx]
    tableau[-1, int_cols] = -1  # use our extra row to store the col==row -> 1
    lengths = np.linalg.norm(tableau, norm, axis=0)
    
    # normalize the columns:
    tableau *= radius
    tableau /= lengths

    # left off: tableau is the wrong size; needs to be full vector or lp needs to be smarter

    # generate the LP to find the hyperplane
    int_basis = [integer_idx[base] for base in basis]
    w, b = compute_hyperplane_via_lp(target - current, radius, tableau, int_basis, col_to_var, integer_idx)
    print("   CUT:", w, current, b)

    # add the cut:
    if b >= 0:
        relaxed.addConstr(w @ (integer_vars - current) >= b)
    else:
        relaxed.addConstr(w @ (integer_vars - current) <= b)
    return True

run_cuts_to_nearest_int(el.get_instances().values(), add_lp_ball_cut)

Set parameter Presolve to value 0
Set parameter Heuristics to value 0
Set parameter Cuts to value 0
Set parameter Presolve to value 0
Set parameter Heuristics to value 0
Set parameter Cuts to value 0
Set parameter Presolve to value 0
Set parameter Heuristics to value 0
Set parameter Cuts to value 0
Set parameter Presolve to value 0
Set parameter Heuristics to value 0
Set parameter Cuts to value 0
Running model: 2D from bottom
   Negated 2 constraints on 2D from bottom
   Relaxed 2 variables on 2D from bottom
   Gap to target: 0.5555555555555558 : [1.22222222 2.33333333] to [1. 2.]
   CUT: [-0.1 -0.9] [1.22222222 2.33333333] 0.2777777777777777
   Gap to target: 0.11111111111111105 : [0.94444444 2.05555556] to [1. 2.]
   CUT: [ 0.22222222 -0.77777778] [0.94444444 2.05555556] 0.0308641975308643
   Gap to target: 0.08888888888888968 : [1.04444444 2.04444444] to [1. 2.]
   CUT: [ 0. -1.] [1.04444444 2.04444444] 0.019753086419753263
   Gap to target: 0.04938271604938271 : [0.97530864 2.02469

In [4]:
def run_cuts_to_relaxed_sol(instances, cut_function, loops=8):
    for instance in instances:
        model: gp.Model = instance.as_gurobi_model()
        print("Running model:", model.ModelName)
        model.params.LogToConsole = 0
        gu.standardize_lt_to_gt(model)

        exact_model = model.copy()
        exact_model.update()
        exact_vars = gp.MVar.fromlist([v for v in exact_model.getVars() if v.VType != 'C'])
        z = exact_model.addMVar(exact_vars.shape, lb=-gp.GRB.INFINITY)
        exact_model.setObjective(z.sum(), gp.GRB.MINIMIZE)

        int_vars, int_idx = gu.relax_int_or_bin_to_continuous(model)
        for i in range(loops):
            model.optimize()
            if model.Status != gp.GRB.OPTIMAL:
                print("   FAILED! Status:", model.Status)
                break

            relaxed_x = int_vars.X
            ca = exact_model.addConstr(z >= exact_vars - relaxed_x)
            cb = exact_model.addConstr(z >= relaxed_x - exact_vars)
            exact_model.optimize()
            assert exact_model.Status == gp.GRB.OPTIMAL

            if not cut_function(model, exact_vars.X, int_vars, int_idx):
                print("   Final score of relaxed:", model.getObjective().getValue())
                break

            exact_model.remove(ca)
            exact_model.remove(cb)

# test the cuts on simple examples:
run_cuts_to_relaxed_sol(el.get_instances().values(), add_balas_ball_cut)

Set parameter Presolve to value 0
Set parameter Heuristics to value 0
Set parameter Cuts to value 0
Set parameter Presolve to value 0
Set parameter Heuristics to value 0
Set parameter Cuts to value 0
Set parameter Presolve to value 0
Set parameter Heuristics to value 0
Set parameter Cuts to value 0
Set parameter Presolve to value 0
Set parameter Heuristics to value 0
Set parameter Cuts to value 0
Running model: 2D from bottom
   Negated 2 constraints on 2D from bottom
   Relaxed 2 variables on 2D from bottom
   Gap to target: 0.5555555555555558 : [1.22222222 2.33333333] to [1. 2.]
   Gap to target: 0.11111111111111105 : [0.94444444 2.05555556] to [1. 2.]
   Gap to target: 0.0888888888888888 : [1.04444444 2.04444444] to [1. 2.]
   Gap to target: 0.16000000000000147 : [1.12444444 2.03555556] to [1. 2.]
   Gap to target: 0.04938271604939248 : [0.97530864 2.02469136] to [1. 2.]
   Gap to target: 0.04768211920530252 : [1.02384106 2.02384106] to [1. 2.]
   Gap to target: 0.09372220516646879 

In [13]:
import importlib
importlib.reload(ml)
miplib_instances = ml.get_instances()
miplib_subset = [miplib_instances['air05'], miplib_instances['markshare2']]  # mas76
run_cuts_to_relaxed_sol(miplib_subset, add_balas_ball_cut)

Read MPS format model from file mip2017_benchmark/revised-submissions/miplib2010_publically_available/instances/air05.mps.gz
Reading time = 0.02 seconds
air05: 426 rows, 7195 columns, 52121 nonzeros
Running model: air05
   Negated 0 constraints on air05
   Relaxed 7195 variables on air05
   Gap to target: 59.472052000350146 : [0.         0.         0.20409116 0.         0.88092007 0.
 0.67214538] to [ 0. -0.  1. -0.  1. -0.  1.]
   Gap to target: 59.68320337660056 : [0.         0.         0.17510936 0.         0.87843237 0.
 0.65546935] to [-0. -0.  1. -0.  1. -0.  1.]
   Gap to target: 59.65125787644668 : [0.         0.         0.23964691 0.08401053 0.85165929 0.
 0.6058634 ] to [ 0. -0.  1. -0.  1. -0.  1.]
   Gap to target: 60.59698282483152 : [0.         0.         0.22366116 0.0503974  0.8333121  0.
 0.62347583] to [0. 0. 1. 0. 1. 0. 1.]
   Gap to target: 61.597094415243745 : [0.         0.         0.20892777 0.02564031 0.88984728 0.
 0.70063238] to [ 0. -0.  1. -0.  1. -0.  1.]
 